In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
drivers_df = spark.table(bronze_table)
display(drivers_df)

In [0]:
from pyspark.sql import functions as F

In [0]:
drivers_dropped_df = drivers_df.drop(F.col("url"))

In [0]:
drivers_renamed_df = (
    drivers_dropped_df
        .withColumnsRenamed({
            "driverId": "driver_id",
            "dateOfBirth": "date_of_birth"
        })
)

In [0]:
drivers_concatenated_df = (
    drivers_renamed_df
        .withColumn("driver_name"
                    , F.initcap(
                        F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))
                    )
        )
        .drop("name")
)
display(drivers_concatenated_df)

In [0]:
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])
display(drivers_distinct_df)

In [0]:
drivers_final_df = (
    drivers_distinct_df
        .withColumn('nationality', F.initcap(F.col("nationality")))
)
display(drivers_final_df)

In [0]:
write_to_silver(
    input_df=drivers_final_df,
    target_table=silver_table,
    merge_condition="t.driver_id = s.driver_id",
    columns_to_update=[
        "driver_name",
        "date_of_birth",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))